In [1]:
# !pip install transformers datasets torch pandas

In [2]:
import os
import pandas as pd
from datasets import Dataset
from transformers import (
    T5Tokenizer,
    T5ForConditionalGeneration,
    Trainer,
    TrainingArguments
)

c:\Models\LLM-From-Scratch\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
train_df = pd.read_csv("../data/processed/train.csv")
val_df = pd.read_csv("../data/processed/validation.csv")

train_df = train_df.dropna(subset=["question", "answer"])
val_df = val_df.dropna(subset=["question", "answer"])

train_df = train_df.head(500)
val_df = val_df.head(100)

print(train_df.shape)
print(val_df.shape)

train_df.head()

(500, 2)
(69, 2)


,question,answer
0,what does memantine look like,"Color - ORANGE, Shape - CAPSULE (biconvex), Sc..."
1,how does a 20 mcg bedford norton transdermal p...,25 mcg fentanyl/hr = 40 mg oral / 20 mg IV oxy...
2,what mg norco comes in,... NORCO® 5/325 ... NORCO® 7.5/325 ... NORCO®...
3,vyvanse 10 what is all in this pill is it safe,The following adverse reactions are discussed ...
4,dtap/tdap/td vaccines how often is this due,"Routine Vaccination of Infants and Children, A..."


In [4]:
train_df["input_text"] = "question: " + train_df["question"]
train_df["target_text"] = train_df["answer"]

val_df["input_text"] = "question: " + val_df["question"]
val_df["target_text"] = val_df["answer"]

train_dataset = Dataset.from_pandas(train_df[["input_text", "target_text"]])
val_dataset = Dataset.from_pandas(val_df[["input_text", "target_text"]])

In [5]:
model_name = "google/flan-t5-base"

tokenizer = T5Tokenizer.from_pretrained(model_name)
model = T5ForConditionalGeneration.from_pretrained(model_name)

Loading weights: 100%|██████████| 282/282 [00:00<00:00, 1550.58it/s]
[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


In [6]:
# !pip install sentencepiece

In [7]:
max_input_length = 128
max_target_length = 256

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=max_input_length,
        truncation=True,
        padding="max_length"
    )

    labels = tokenizer(
        examples["target_text"],
        max_length=max_target_length,
        truncation=True,
        padding="max_length"
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)

train_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

val_dataset.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

Map: 100%|██████████| 69/69 [00:00<00:00, 2023.43 examples/s]


In [8]:
training_args = TrainingArguments(
    output_dir="../models/t5",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-4,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=10
)

In [9]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

c:\Models\LLM-From-Scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss
1,0.872861,1.036889
2,0.983040,1.007481
3,0.659522,1.012567


Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.55s/it]
c:\Models\LLM-From-Scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:01<00:00,  1.71s/it]
c:\Models\LLM-From-Scratch\.venv\Lib\site-packages\torch\utils\data\dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
Writing model shards: 100%|██████████| 1/1 [00:09<00:00,  9.56s/it]


TrainOutput(global_step=750, training_loss=1.2215664329528808, metrics={'train_runtime': 9165.2304, 'train_samples_per_second': 0.164, 'train_steps_per_second': 0.082, 'total_flos': 256784007168000.0, 'train_loss': 1.2215664329528808, 'epoch': 3.0})

In [15]:
model.save_pretrained("../models/t5")
tokenizer.save_pretrained("../models/t5")

print("T5 modeli kaydedildi.")

Writing model shards: 100%|██████████| 1/1 [00:13<00:00, 13.75s/it]


T5 modeli kaydedildi.


In [16]:
def generate_answer(question):
    input_text = "question: " + question

    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=128
    )

    outputs = model.generate(
        **inputs,
        max_length=128,
        num_beams=5,
        do_sample=True,
        temperature=0.7,
        top_k=50,
        top_p=0.95,
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
generate_answer("Does ibuprofen contain aspirin?")

'Aspirin is in a class of medications called antacids. It works by decreasing the amount of aspirin in the blood.'

In [18]:
generate_answer("Can I take ibuprofen with food?")

'Taking ibuprofen with food can lead to serious side effects. If you take ibuprofen with food, you may experience side effects such as stomach aches, nausea, vomiting, and diarrhea. If you take ibuprofen with food, you may experience side effects such as stomach aches, nausea, vomiting, and diarrhea.'